# Take the frame based detections and turn them into movement trajectories
(For speed, uses multiprocessing and breaks up observations into 10 overlapping temporal groups to process seperately since tracks are much shorter in time than the total length of the obervations.)

In [1]:
import glob
import os

import matplotlib.pyplot as plt
from multiprocessing import Pool
import numpy as np

import bat_functions as kbf

C:\Users\Edward\anaconda3\envs\Eidolon\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\Edward\anaconda3\envs\Eidolon\lib\site-packages\numpy\.libs\libopenblas.GK7GX5KEQ4F6UYO3P26ULGBQYHGQO7J4.gfortran-win_amd64.dll
C:\Users\Edward\anaconda3\envs\Eidolon\lib\site-packages\numpy\.libs\libopenblas.PYQHXLVVQ7VESDPUVUADXEVJOBGHJPAY.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [8]:
base_folder = "D:/kasanka-bats"
# Which days to track
days = ["20211221", "20211201", "20211207", "20211215"]
camera_folders = []

for day in days:
    camera_folders.extend(sorted(glob.glob(os.path.join(base_folder, day, '*'))))
print(camera_folders)

['D:/kasanka-bats\\20211221\\BBC2', 'D:/kasanka-bats\\20211221\\ChinyangaliB', 'D:/kasanka-bats\\20211221\\FibweManagement', 'D:/kasanka-bats\\20211221\\FibweParking', 'D:/kasanka-bats\\20211221\\KKCamera', 'D:/kasanka-bats\\20211221\\NotChinyangaliLocC', 'D:/kasanka-bats\\20211221\\NotMusolaParking', 'D:/kasanka-bats\\20211221\\NotMusolaPath', 'D:/kasanka-bats\\20211201\\BBC', 'D:/kasanka-bats\\20211201\\Chinyangali', 'D:/kasanka-bats\\20211201\\FibweParking', 'D:/kasanka-bats\\20211201\\FibwePublic', 'D:/kasanka-bats\\20211201\\MusolaParking', 'D:/kasanka-bats\\20211201\\MusolaPath', 'D:/kasanka-bats\\20211201\\NotChinyangaliLocB', 'D:/kasanka-bats\\20211207\\BBC', 'D:/kasanka-bats\\20211207\\Chinyangali', 'D:/kasanka-bats\\20211207\\FibweParking', 'D:/kasanka-bats\\20211207\\FibwePublic', 'D:/kasanka-bats\\20211207\\MusolaParking', 'D:/kasanka-bats\\20211207\\MusolaPath', 'D:/kasanka-bats\\20211207\\NotChinyangaliLocB', 'D:/kasanka-bats\\20211215\\BBC', 'D:/kasanka-bats\\20211215\\C

In [9]:
n_camera_folders = []
for folder in camera_folders:
    tracks_file = os.path.join(folder, "raw_tracks.npy")
    if os.path.exists(tracks_file):
        # Skip videos that have already been tracked
        continue
    else:
        n_camera_folders.append(folder)

print("Videos to track...")
print(*n_camera_folders, sep='\n')

Videos to track...
D:/kasanka-bats\20211221\BBC2
D:/kasanka-bats\20211221\ChinyangaliB
D:/kasanka-bats\20211221\FibweManagement
D:/kasanka-bats\20211221\FibweParking
D:/kasanka-bats\20211221\KKCamera
D:/kasanka-bats\20211221\NotChinyangaliLocC
D:/kasanka-bats\20211221\NotMusolaParking
D:/kasanka-bats\20211221\NotMusolaPath
D:/kasanka-bats\20211201\BBC
D:/kasanka-bats\20211201\Chinyangali
D:/kasanka-bats\20211201\FibweParking
D:/kasanka-bats\20211201\FibwePublic
D:/kasanka-bats\20211201\MusolaParking
D:/kasanka-bats\20211201\MusolaPath
D:/kasanka-bats\20211201\NotChinyangaliLocB
D:/kasanka-bats\20211207\BBC
D:/kasanka-bats\20211207\Chinyangali
D:/kasanka-bats\20211207\FibweParking
D:/kasanka-bats\20211207\FibwePublic
D:/kasanka-bats\20211207\MusolaParking
D:/kasanka-bats\20211207\MusolaPath
D:/kasanka-bats\20211207\NotChinyangaliLocB
D:/kasanka-bats\20211215\BBC
D:/kasanka-bats\20211215\Chinyangali
D:/kasanka-bats\20211215\FibweManagement
D:/kasanka-bats\20211215\FibweParking
D:/kasanka

In [10]:
def track(camera_dict):
    camera_folder = camera_dict['camera_folder']
    first_frame = camera_dict['first_frame']
    max_frame = camera_dict['max_frame']
    print(f"{os.path.basename(camera_folder)} begun.")
    contours_files = sorted(
        glob.glob(os.path.join(camera_folder, 'contours-compressed-*.npy'))
    )
    if contours_files:
        contours_files = contours_files[1:]
        centers = np.load(os.path.join(camera_folder, 'centers.npy'), allow_pickle=True)
        sizes = np.load(os.path.join(camera_folder, 'size.npy'), allow_pickle=True)
        tracks_file = os.path.join(camera_folder, f'first_frame_{first_frame}_max_val_{max_frame}_raw_tracks.npy')
        raw_tracks = kbf.find_tracks(first_frame, centers, contours_files=contours_files, 
                                     sizes_list=sizes, tracks_file=tracks_file,
                                     max_frame=max_frame)
    else:
        print("Missing contour files.")

In [11]:
camera_dicts = []
for camera_folder in n_camera_folders:
    # To speed up processing, detections found in each observation are split
    # into 10 groups by time with 15 seconds of overlap in each group
    centers_file = os.path.join(camera_folder, 'centers.npy')
    centers = np.load(centers_file, allow_pickle=True)
    max_vals = np.linspace(0, len(centers), 10, dtype=int)[1:].tolist()
    max_vals[-1] = None
    min_vals = np.linspace(0, len(centers), 10, dtype=int)[:-1]
    # 15 second overlap
    min_vals[1:] = min_vals[1:] - 450
    for min_val, max_val in zip(min_vals, max_vals):
        min_val = np.max([min_val, 0])
        camera_dict = {'camera_folder': camera_folder,
                       'first_frame': min_val,
                       'max_frame': max_val}
        if max_val is None:
            tracks_basename = f'first_frame_{min_val:06d}_max_val_{max_val}_raw_tracks.npy'
        else:
            tracks_basename = f'first_frame_{min_val:06d}_max_val_{max_val:06d}_raw_tracks.npy'
        tracks_file = os.path.join(camera_folder, tracks_basename)
        if not os.path.exists(tracks_file):
            print(tracks_file)
            camera_dicts.append(camera_dict)
        

D:/kasanka-bats\20211221\BBC2\first_frame_000000_max_val_007069_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_006619_max_val_014139_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_013689_max_val_021209_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_020759_max_val_028279_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_027829_max_val_035349_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_034899_max_val_042419_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_041969_max_val_049489_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_049039_max_val_056559_raw_tracks.npy
D:/kasanka-bats\20211221\BBC2\first_frame_056109_max_val_None_raw_tracks.npy
D:/kasanka-bats\20211221\ChinyangaliB\first_frame_000000_max_val_009620_raw_tracks.npy
D:/kasanka-bats\20211221\ChinyangaliB\first_frame_009170_max_val_019240_raw_tracks.npy
D:/kasanka-bats\20211221\ChinyangaliB\first_frame_018790_max_val_028860_raw_tracks.npy
D:/kasanka-bats\20211221\Chiny

D:/kasanka-bats\20211201\FibwePublic\first_frame_000000_max_val_005365_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_004915_max_val_010731_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_010281_max_val_016097_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_015647_max_val_021463_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_021013_max_val_026829_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_026379_max_val_032195_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_031745_max_val_037561_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_037111_max_val_042927_raw_tracks.npy
D:/kasanka-bats\20211201\FibwePublic\first_frame_042477_max_val_None_raw_tracks.npy
D:/kasanka-bats\20211201\MusolaParking\first_frame_000000_max_val_006223_raw_tracks.npy
D:/kasanka-bats\20211201\MusolaParking\first_frame_005773_max_val_012446_raw_tracks.npy
D:/kasanka-bats\20211201\MusolaParking\first_frame_0

D:/kasanka-bats\20211215\BBC\first_frame_000000_max_val_004292_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_003842_max_val_008584_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_008134_max_val_012876_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_012426_max_val_017168_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_016718_max_val_021460_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_021010_max_val_025752_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_025302_max_val_030044_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_029594_max_val_034336_raw_tracks.npy
D:/kasanka-bats\20211215\BBC\first_frame_033886_max_val_None_raw_tracks.npy
D:/kasanka-bats\20211215\Chinyangali\first_frame_000000_max_val_006413_raw_tracks.npy
D:/kasanka-bats\20211215\Chinyangali\first_frame_005963_max_val_012826_raw_tracks.npy
D:/kasanka-bats\20211215\Chinyangali\first_frame_012376_max_val_019240_raw_tracks.npy
D:/kasanka-bats\20211215\Chinyangali\first

In [12]:
#with Pool(processes=5) as pool:
#    pool.map(track, camera_dicts)
for camera_dict in camera_dicts:
    track(camera_dict)

BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-01.npy
frame 0 processed.
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-02.npy
frame 10000 processed.
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-04.npy
frame 20000 processed.
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-05.npy
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-07.npy
frame 30000 processed.
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-09.npy
frame 40000 processed.
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-10.npy
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-12.npy
frame 50000 processed.
BBC2 final save.
BBC2 begun.
using D:/kasanka-bats\20211221\BBC2\contours-compressed-14.npy
frame 60000 processed.
BBC2 final save.
ChinyangaliB be

frame 50000 processed.
NotMusolaParking final save.
NotMusolaParking begun.
using D:/kasanka-bats\20211221\NotMusolaParking\contours-compressed-12.npy
frame 60000 processed.
NotMusolaParking final save.
NotMusolaParking begun.
using D:/kasanka-bats\20211221\NotMusolaParking\contours-compressed-14.npy
NotMusolaParking final save.
NotMusolaPath begun.
using D:/kasanka-bats\20211221\NotMusolaPath\contours-compressed-01.npy
frame 0 processed.
frame 10000 processed.
NotMusolaPath final save.
NotMusolaPath begun.
using D:/kasanka-bats\20211221\NotMusolaPath\contours-compressed-02.npy
frame 20000 processed.
NotMusolaPath final save.
NotMusolaPath begun.
using D:/kasanka-bats\20211221\NotMusolaPath\contours-compressed-04.npy
frame 30000 processed.
frame 40000 processed.
NotMusolaPath final save.
NotMusolaPath begun.
using D:/kasanka-bats\20211221\NotMusolaPath\contours-compressed-05.npy
frame 40000 processed.
frame 50000 processed.
NotMusolaPath final save.
NotMusolaPath begun.
using D:/kasank

using D:/kasanka-bats\20211201\MusolaPath\contours-compressed-14.npy
frame 80000 processed.
MusolaPath final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211201\NotChinyangaliLocB\contours-compressed-01.npy
frame 0 processed.
frame 10000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211201\NotChinyangaliLocB\contours-compressed-02.npy
frame 20000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211201\NotChinyangaliLocB\contours-compressed-04.npy
frame 30000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211201\NotChinyangaliLocB\contours-compressed-05.npy
frame 40000 processed.
frame 50000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211201\NotChinyangaliLocB\contours-compressed-07.npy
frame 60000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211201\N

frame 90000 processed.
MusolaPath final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211207\NotChinyangaliLocB\contours-compressed-01.npy
frame 0 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211207\NotChinyangaliLocB\contours-compressed-02.npy
frame 10000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211207\NotChinyangaliLocB\contours-compressed-04.npy
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211207\NotChinyangaliLocB\contours-compressed-05.npy
frame 20000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211207\NotChinyangaliLocB\contours-compressed-07.npy
frame 30000 processed.
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211207\NotChinyangaliLocB\contours-compressed-09.npy
NotChinyangaliLocB final save.
NotChinyangaliLocB begun.
using D:/kasanka-bats\20211207\NotCh

MusolaParking final save.
MusolaParking begun.
using D:/kasanka-bats\20211215\MusolaParking\contours-compressed-04.npy
MusolaParking final save.
MusolaParking begun.
using D:/kasanka-bats\20211215\MusolaParking\contours-compressed-05.npy
frame 20000 processed.
MusolaParking final save.
MusolaParking begun.
using D:/kasanka-bats\20211215\MusolaParking\contours-compressed-07.npy
MusolaParking final save.
MusolaParking begun.
using D:/kasanka-bats\20211215\MusolaParking\contours-compressed-09.npy
frame 30000 processed.
MusolaParking final save.
MusolaParking begun.
using D:/kasanka-bats\20211215\MusolaParking\contours-compressed-10.npy
MusolaParking final save.
MusolaParking begun.
using D:/kasanka-bats\20211215\MusolaParking\contours-compressed-12.npy
frame 40000 processed.
MusolaParking final save.
MusolaParking begun.
using D:/kasanka-bats\20211215\MusolaParking\contours-compressed-14.npy
MusolaParking final save.


### Now connect the 10 sections of the observation that were tracked seperately together

In [13]:
def combine_overlapping_tracks(observation_folder, first_group=0, last_group=None, save=False):
    track_files = glob.glob(os.path.join(observation_folder, 'first_frame*.npy'))
    track_files = sorted(track_files, key=lambda f: int(f.split('_')[-6]))

    track_groups = []
    for file in track_files:
        track_groups.append(np.load(file, allow_pickle=True))
        
        
    for track_file in track_files:
        print(os.path.basename(track_file))
        
    first_overlap_frames = [int(f.split('_')[-6]) for f in track_files[1:]]
    first_overlap_frames.append(None)
    print(first_overlap_frames)
        
    all_tracks = []

    total_tracks = 0
    for group_ind, track_group in enumerate(track_groups[first_group:last_group]):
        if group_ind >= len(track_groups) -1:
            for track in track_group:
                if type(track['track']) == list:
                    track['track'] = np.stack(track['track'])
                    track['pos_index'] = np.stack(track['pos_index'])
                    if 'size' in track:
                        track['size'] = np.stack(track['size'])
                all_tracks.append(track)
            total_tracks += len(track_group)
            break

        for track_ind, track in enumerate(track_group):
            if track['first_frame'] < first_overlap_frames[first_group + group_ind]:
                all_tracks.append(track)


    all_tracks_file = os.path.join(observation_folder, 'raw_tracks.npy')
    if save:
        np.save(all_tracks_file, all_tracks)
        print('saved')

In [14]:
observation_folders = []
for folder in camera_folders:
    if not os.path.exists(os.path.join(folder, 'raw_tracks.npy')):
        observation_folders.append(folder)

In [15]:
for folder in observation_folders:
    combine_overlapping_tracks(folder, save=True)

first_frame_0_max_val_7069_raw_tracks.npy
first_frame_6619_max_val_14139_raw_tracks.npy
first_frame_13689_max_val_21209_raw_tracks.npy
first_frame_20759_max_val_28279_raw_tracks.npy
first_frame_27829_max_val_35349_raw_tracks.npy
first_frame_34899_max_val_42419_raw_tracks.npy
first_frame_41969_max_val_49489_raw_tracks.npy
first_frame_49039_max_val_56559_raw_tracks.npy
first_frame_56109_max_val_None_raw_tracks.npy
[6619, 13689, 20759, 27829, 34899, 41969, 49039, 56109, None]
saved
first_frame_0_max_val_9620_raw_tracks.npy
first_frame_9170_max_val_19240_raw_tracks.npy
first_frame_18790_max_val_28860_raw_tracks.npy
first_frame_28410_max_val_38480_raw_tracks.npy
first_frame_38030_max_val_48100_raw_tracks.npy
first_frame_47650_max_val_57720_raw_tracks.npy
first_frame_57270_max_val_67340_raw_tracks.npy
first_frame_66890_max_val_76960_raw_tracks.npy
first_frame_76510_max_val_None_raw_tracks.npy
[9170, 18790, 28410, 38030, 47650, 57270, 66890, 76510, None]
saved
first_frame_0_max_val_7875_raw_t

MemoryError: 